# 04 - Custom Evaluators

Built-in evaluators cover common questions such as correctness and helpfulness. Create a custom evaluator when your application has a specific requirement that the built-in catalog does not measure directly.

You will:

1. Turn one application requirement into a clear evaluation question.
2. Configure an LLM judge that returns pass or fail.
3. Test a rule-based evaluator locally.
4. Compare a judge with human decisions before trusting its scores.

**Estimated time:** 45-60 minutes  
**Creates AWS resources:** Optional custom evaluator and Lambda resources.

## 1. Start with one clear question

A useful evaluator is easy to explain. Prefer:

- one requirement per evaluator
- pass/fail when the real decision is also pass/fail
- a clear definition of success, including important edge cases
- a predictable score and label format
- examples that people have already labeled

For example, avoid one evaluator that scores "helpfulness, accuracy, clarity, professionalism, and completeness" all at once. If its score changes, you will not know which behavior changed. Several focused evaluators are easier to understand and improve.

## 2. Focused LLM-as-a-Judge

The first evaluator answers one question: **Does the response agree with the supplied city facts?**

It passes when the numbers and comparisons are correct. It ignores writing style, answer length, and tone because those belong to other evaluators.

In [ ]:
FACTUAL_CONSISTENCY_INSTRUCTIONS = '''
Evaluate only factual consistency with the supplied reference information.

PASS when every city fact, number, and comparison in the assistant response
agrees with the reference information in {context}. Reasonable rounding is
allowed when the user asks for an approximation.

FAIL when the response gives a wrong number, reverses a comparison, substitutes
a different city or state, invents data for a missing city, or does not answer
the factual question.

Ignore writing style, tone, and verbosity. Return the configured rating label
and a concise explanation tied to this criterion only.
'''.strip()

print(FACTUAL_CONSISTENCY_INSTRUCTIONS)

The next cell can add this definition to the local AgentCore project. The judge model must be available in your AWS Region, so replace the example model or inference profile if needed.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from src.workshop_utils import MODULE_ROOT, run_cli, run_cli_json

JUDGE_MODEL = os.getenv(
    "AGENTCORE_JUDGE_MODEL_ID",
    "global.anthropic.claude-sonnet-4-6",
)
CREATE_LLM_EVALUATOR = False

config_path = MODULE_ROOT / "agentcore" / "agentcore.json"
project_config = json.loads(config_path.read_text())
existing_names = {
    item["name"] for item in project_config.get("evaluators", [])
}

if CREATE_LLM_EVALUATOR and "FactualConsistency" not in existing_names:
    result = run_cli(
        "add",
        "evaluator",
        "--name",
        "FactualConsistency",
        "--level",
        "TRACE",
        "--model",
        JUDGE_MODEL,
        "--instructions",
        FACTUAL_CONSISTENCY_INSTRUCTIONS,
        "--rating-scale",
        "pass-fail",
    )
    print(result.stdout.strip())
    print("Evaluator added locally. Run agentcore deploy to create it.")
else:
    print(
        "Set CREATE_LLM_EVALUATOR=True to add the evaluator, "
        "or keep reading to unit-test a code evaluator locally."
    )

Adding the evaluator updates the local `agentcore.json` file. Deploying then creates the evaluator in AWS. You can read and improve the instructions locally before creating any resource.

The next cell keeps deployment and evaluation separate. To try the evaluator, provide the session ID from a CityAnalyst request that has already produced trace data.

In [ ]:
DEPLOY_LLM_EVALUATOR = False
RUN_LLM_EVALUATOR = False
LLM_EVALUATION_SESSION_ID = ""

if DEPLOY_LLM_EVALUATOR:
    diff = run_cli("deploy", "--target", "default", "--diff")
    print(diff.stdout.strip())
    llm_deploy_result = run_cli_json(
        "deploy", "--target", "default", "--yes"
    )
    print("FactualConsistency was deployed to the default target.")

if RUN_LLM_EVALUATOR:
    if not LLM_EVALUATION_SESSION_ID:
        raise ValueError("Set LLM_EVALUATION_SESSION_ID before running the evaluator.")
    llm_eval_result = run_cli_json(
        "run",
        "eval",
        "--runtime",
        "CityAnalyst",
        "--evaluator",
        "FactualConsistency",
        "--session-id",
        LLM_EVALUATION_SESSION_ID,
        "--expected-response",
        "Seattle population is 780995.",
    )
    llm_run = llm_eval_result.get("run", llm_eval_result)
    llm_summary = []
    llm_details = []
    for evaluator_result in llm_run.get("results", []):
        scores = evaluator_result.get("sessionScores", [])
        llm_summary.append(
            {
                "Evaluator": evaluator_result.get("evaluator"),
                "Aggregate score": evaluator_result.get("aggregateScore"),
                "Results": len(scores),
                "Errors": sum(
                    bool(item.get("errorCode") or item.get("errorMessage"))
                    for item in scores
                ),
            }
        )
        for score in scores:
            llm_details.append(
                {
                    "target": (
                        score.get("traceId")
                        or score.get("sessionId")
                        or "Not available"
                    ),
                    "value": score.get("value"),
                    "label": score.get("label"),
                    "detail": (
                        score.get("errorMessage")
                        or score.get("errorCode")
                        or score.get("explanation")
                        or "No explanation was returned."
                    ),
                }
            )

    if llm_summary:
        display(pd.DataFrame(llm_summary))
    else:
        print("No evaluator results were returned for this session.")
    for item in llm_details:
        display(
            Markdown(
                f"**Result for:** `{item['target']}`  \n"
                f"**Score:** `{item['value'] if item['value'] is not None else 'Not available'}` | "
                f"**Label:** `{item['label'] or 'Not available'}`\n\n"
                f"{item['detail']}"
            )
        )

## 3. Rule-based code evaluator

The second evaluator answers a different question: **Does the final response use the required JSON structure?**

Code is a better fit here because JSON rules are exact. The same input always produces the same result, and a failure message can point directly to the missing or invalid field.

In [ ]:
from bedrock_agentcore.evaluation.custom_code_based_evaluators import EvaluatorInput
from evaluators.response_format_evaluator import (
    handler,
    validate_response_schema,
)

valid_response = json.dumps(
    {
        "answer": "Seattle has 780995 residents.",
        "cities": [
            {
                "city": "Seattle",
                "state": "WA",
                "population": 780995,
                "land_area_mi2": 83.8,
                "density_per_mi2": 9319.7,
            }
        ],
        "tools_used": ["lookup_city"],
    }
)
invalid_response = "Seattle has 780995 residents."

schema_checks = []
for example_name, response_text in (
    ("Valid JSON response", valid_response),
    ("Plain-text response", invalid_response),
):
    passed, explanation = validate_response_schema(response_text)
    schema_checks.append(
        {
            "Example": example_name,
            "Passed": passed,
            "Explanation": explanation,
        }
    )

display(pd.DataFrame(schema_checks))

AgentCore wraps the evaluator so it can receive Lambda events in AWS. For a local test, `handler.unwrapped` calls the evaluation logic directly. This lets us test the scoring rule without deploying a Lambda function first.

In [ ]:
fixture_spans = json.loads(Path("data/sample_spans.json").read_text())
evaluator_input = EvaluatorInput(
    evaluation_level="TRACE",
    session_spans=fixture_spans,
    target_trace_id=fixture_spans[0]["traceId"],
    evaluator_id="local-response-format",
    evaluator_name="ResponseFormat",
)

local_result = handler.unwrapped(evaluator_input, context=None)
display(
    pd.DataFrame(
        [
            {
                "Fixture": "Valid structured response",
                "Score": local_result.value,
                "Label": local_result.label,
                "Explanation": local_result.explanation,
            }
        ]
    )
)

In [ ]:
failing_spans = json.loads(Path("data/sample_spans.json").read_text())
failing_spans[-1]["attributes"]["gen_ai.response.content"] = (
    "Seattle has 780995 residents."
)
failing_input = EvaluatorInput(
    evaluation_level="TRACE",
    session_spans=failing_spans,
    target_trace_id=failing_spans[0]["traceId"],
    evaluator_id="local-response-format",
    evaluator_name="ResponseFormat",
)

failing_result = handler.unwrapped(failing_input, context=None)
display(
    pd.DataFrame(
        [
            {
                "Fixture": "Plain-text response",
                "Score": failing_result.value,
                "Label": failing_result.label,
                "Explanation": failing_result.explanation,
            }
        ]
    )
)

## 4. Register the code evaluator

There are two common ways to make a code evaluator available to AgentCore:

1. Ask the CLI to create and manage the Lambda resources.
2. Register a Lambda function that your team already manages, as shown below.

Run local examples first. Deploying should publish a rule you already understand, rather than being the first time you discover whether the rule works.

In [ ]:
REGISTER_CODE_EVALUATOR = False
DEPLOY_CODE_EVALUATOR = False
CODE_EVALUATOR_LAMBDA_ARN = ""

if REGISTER_CODE_EVALUATOR and "ResponseFormat" not in existing_names:
    if not CODE_EVALUATOR_LAMBDA_ARN:
        raise ValueError("Set CODE_EVALUATOR_LAMBDA_ARN before registration.")
    result = run_cli(
        "add",
        "evaluator",
        "--name",
        "ResponseFormat",
        "--level",
        "TRACE",
        "--type",
        "code-based",
        "--lambda-arn",
        CODE_EVALUATOR_LAMBDA_ARN,
        "--timeout",
        "30",
    )
    print(result.stdout.strip())
elif REGISTER_CODE_EVALUATOR:
    print("ResponseFormat is already present in agentcore.json.")

if DEPLOY_CODE_EVALUATOR:
    code_deploy_result = run_cli_json(
        "deploy", "--target", "default", "--yes"
    )
    print("ResponseFormat was deployed to the default target.")

## 5. Validate the judge, not just the agent

An LLM judge can produce a clear explanation and still make the wrong decision. Test the judge against examples that people have already labeled.

The checked-in file contains eight factual-consistency examples: four people labeled `pass` and four labeled `fail`. These human labels are the answer key used to check the judge.

In [ ]:
validation_examples = [
    json.loads(line)
    for line in Path("data/judge_validation.jsonl").read_text().splitlines()
    if line.strip()
]
pd_rows = [
    {
        "Example": item["id"],
        "Human label": item["human_label"],
        "Question": item["question"],
        "Response to evaluate": item["response"],
        "Why": item["rationale"],
    }
    for item in validation_examples
]

validation_df = pd.DataFrame(pd_rows)
display(validation_df)
display(
    validation_df["Human label"]
    .value_counts()
    .rename_axis("Human label")
    .reset_index(name="Examples")
)

In [ ]:
def judge_scorecard(records):
    evaluated = [
        item for item in records
        if item.get("judge_label") in {"pass", "fail"}
    ]
    if not evaluated:
        raise ValueError("Add judge_label values before computing the scorecard.")

    true_pass = [item for item in evaluated if item["human_label"] == "pass"]
    true_fail = [item for item in evaluated if item["human_label"] == "fail"]
    missing_classes = [
        label
        for label, records_for_label in (("pass", true_pass), ("fail", true_fail))
        if not records_for_label
    ]
    if missing_classes:
        raise ValueError(
            "Calibration requires at least one human-labeled pass and fail example. "
            f"Missing: {', '.join(missing_classes)}."
        )
    accuracy = sum(
        item["human_label"] == item["judge_label"]
        for item in evaluated
    ) / len(evaluated)
    tpr = sum(item["judge_label"] == "pass" for item in true_pass) / len(true_pass)
    tnr = sum(item["judge_label"] == "fail" for item in true_fail) / len(true_fail)
    return {
        "Examples scored": len(evaluated),
        "Overall agreement": accuracy,
        "Pass examples recognized": tpr,
        "Fail examples recognized": tnr,
    }

print(
    "Run the custom evaluator on each example and store its decision as "
    "judge_label. Then call judge_scorecard(validation_examples)."
)

Checking the judge is often called **calibration**. A practical workflow is:

1. Keep separate examples for writing the instructions, improving them, and testing the final evaluator.
2. Keep very similar questions in the same group so the final test is genuinely new.
3. Read every case where the judge and human label disagree.
4. Improve the evaluator instructions or examples; do not change a correct human label just to improve the score.
5. Run the judge more than once to check whether its decisions are stable.
6. Record overall agreement, the share of pass examples recognized, and the share of fail examples recognized.

The foundational notebook [Evaluating Your Judge](../../Foundational%20Evaluations/02-quality-metrics/03_Evaluating_your_Judge.ipynb) explains this process in more depth.

## 6. Checkpoint

You should now be able to decide:

- built-in or custom
- LLM judge or rule-based code
- session, trace, or tool-call level
- which human-labeled examples are needed to test the evaluator

Continue to [05 - Batch and Online Evaluation](05-batch-and-online-evaluation.ipynb) to use evaluation results in automated checks and ongoing monitoring.